In [23]:
# ============================================================================
# MIXTURE OF EXPERTS (MoE) PHISHING DETECTION SYSTEM
# Thesis Implementation - Professional Version
# ============================================================================

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import csr_matrix
import re

# ============================================================================
# STEP 1: Define URLFeatures Class
# ============================================================================

class URLFeatures(BaseEstimator, TransformerMixin):
    """
    Feature extractor for URL-based phishing detection.
    Extracts structural characteristics from URLs.
    """
    def fit(self, X, y=None):
        return self
    
    def transform(self, urls):
        urls = np.array(urls).reshape(-1)
        feats = np.array([
            [
                len(u),
                u.count('-'),
                u.count('@'),
                u.count('?'),
                u.count('='),
                u.count('.'),
                int(u.startswith("https")),
                int(u.count("//") > 1)
            ]
            for u in urls
        ])
        return csr_matrix(feats)

# ============================================================================
# STEP 2: Define Gating Network Architecture
# ============================================================================

class GatingNetwork(nn.Module):
    """
    Neural network-based gating mechanism for Mixture of Experts.
    Learns optimal weight distribution between URL and Text experts.
    
    Architecture:
        Input Layer: 8 features
        Hidden Layer: 64 neurons with ReLU activation
        Output Layer: 2 expert weights with Softmax normalization
    """
    def __init__(self, input_size=8, hidden_size=64, num_experts=2):
        super(GatingNetwork, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_experts)
        self.softmax = nn.Softmax(dim=1)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        weights = self.softmax(x)
        return weights

# ============================================================================
# STEP 3: Load Expert Models
# ============================================================================

print("Loading Expert Models...")
print("-" * 70)

# Expert 1: URL-based Phishing Detector
URL_MODEL_PATH = r"C:\Users\angelo\Downloads\THESIS\URL_Expert-20251210T060216Z-1-001\URL_Expert\Notebook and Model\url_expert_1.pkl"
expert_1 = joblib.load(URL_MODEL_PATH)
print("Expert 1 (URL-based): Loaded successfully")

# Expert 2: Text-based Phishing Detector (DistilBERT)
TEXT_MODEL_PATH = r"C:\Users\angelo\Downloads\THESIS\distilbert_phishing_model"
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_PATH)
expert_2 = AutoModelForSequenceClassification.from_pretrained(TEXT_MODEL_PATH)
expert_2.eval()
print("Expert 2 (Text-based): Loaded successfully")

# ============================================================================
# STEP 4: Load Trained Gating Network
# ============================================================================

print("\nLoading Trained Gating Network...")
print("-" * 70)

gating_net = GatingNetwork(input_size=8, hidden_size=64, num_experts=2)
gating_net.load_state_dict(torch.load('gating_network.pth'))
gating_net.eval()
print("Gating Network: Loaded successfully")

print("\n" + "=" * 70)
print("MoE System Initialization Complete")
print("=" * 70 + "\n")

# ============================================================================
# STEP 5: Feature Extraction Components
# ============================================================================

# Phishing-indicative phrase dictionary with weighted scores
phrase_dict = {
    'urgent': 0.3,
    'verify account': 0.5,
    'suspended': 0.4,
    'click here': 0.3,
    'confirm your': 0.4,
    'congratulations': 0.3,
    'winner': 0.4,
    'limited time': 0.3,
    'act now': 0.3,
    'security alert': 0.5,
    'claim': 0.3,
    'prize': 0.3,
    'free': 0.2,
    'bonus': 0.2,
}

def preprocess_text(text):
    """
    Preprocess text input by removing URLs and normalizing whitespace.
    
    Args:
        text (str): Raw input text
        
    Returns:
        str: Preprocessed text
    """
    if pd.isna(text) or text == "":
        return ""
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def calculate_phrase_score(text, phrase_dict):
    """
    Calculate phishing phrase score based on suspicious keyword presence.
    
    Args:
        text (str): Input text
        phrase_dict (dict): Dictionary of phrases and their weights
        
    Returns:
        float: Normalized phrase score (0.0 to 1.0)
    """
    if not text:
        return 0.0
    text_lower = text.lower()
    score = 0.0
    for phrase, weight in phrase_dict.items():
        if phrase in text_lower:
            score += weight
    return min(score, 1.0)

def extract_gating_features(text, url, phrase_score):
    """
    Extract features for gating network decision-making.
    
    Features:
        1. URL presence (binary)
        2. Phrase score (0.0 to 1.0)
        3. Message length (word count)
        4. Special character count
        5. Hashtag count
        6. URL count
        7. Capital letter ratio
        8. Embedding summary (placeholder)
    
    Args:
        text (str): Processed text
        url (str): Extracted URL
        phrase_score (float): Pre-calculated phrase score
        
    Returns:
        np.array: Feature vector for gating network
    """
    url_present = 1 if (url and not pd.isna(url) and url != "") else 0
    message_length = len(text.split()) if text else 0
    emoji_count = len(re.findall(r'[^\w\s,]', text)) if text else 0
    hashtag_count = text.count('#') if text else 0
    url_count = len(re.findall(r'http\S+', text)) if text else 0
    
    if text and len(text) > 0:
        capital_ratio = sum(1 for c in text if c.isupper()) / len(text)
    else:
        capital_ratio = 0.0
    
    embedding_summary = 0.0
    
    features = np.array([
        url_present,
        phrase_score,
        message_length,
        emoji_count,
        hashtag_count,
        url_count,
        capital_ratio,
        embedding_summary
    ], dtype=np.float32)
    
    return features

# ============================================================================
# STEP 6: Prediction Function
# ============================================================================

def predict_with_trained_model(text, url):
    """
    Perform phishing detection using Mixture of Experts with learned gating.
    
    Process:
        1. Preprocess input text
        2. Obtain predictions from both experts (URL and Text)
        3. Use gating network to determine expert weights
        4. Combine expert predictions using learned weights
        5. Generate final classification
    
    Args:
        text (str): Message text content
        url (str): URL (if present)
        
    Returns:
        dict: Prediction results including:
            - prediction: Final classification (PHISHING/SAFE)
            - confidence: Prediction confidence (0-100%)
            - url_weight: Weight assigned to URL expert
            - text_weight: Weight assigned to Text expert
            - Individual expert predictions and probabilities
    """
    
    text = preprocess_text(text)
    phrase_score = calculate_phrase_score(text, phrase_dict)
    
    # Obtain URL Expert Prediction
    if url and url.strip():
        try:
            url_df = pd.DataFrame({'url': [url]})
            url_probs = expert_1.predict_proba(url_df)[0]
        except:
            url_probs = np.array([0.5, 0.5])
    else:
        url_probs = np.array([0.5, 0.5])
    
    # Obtain Text Expert Prediction
    if text:
        try:
            inputs = tokenizer(text, return_tensors='pt', padding=True, 
                             truncation=True, max_length=128)
            with torch.no_grad():
                outputs = expert_2(**inputs)
                text_probs = torch.softmax(outputs.logits, dim=1)[0].numpy()
        except:
            text_probs = np.array([0.5, 0.5])
    else:
        text_probs = np.array([0.5, 0.5])
    
    # Compute Gating Weights
    gating_features = extract_gating_features(text, url, phrase_score)
    gating_input = torch.FloatTensor(gating_features).unsqueeze(0)
    
    with torch.no_grad():
        expert_weights = gating_net(gating_input)
    
    # Weighted Combination of Expert Predictions
    final_probs = (expert_weights[0, 0].item() * url_probs + 
                  expert_weights[0, 1].item() * text_probs)
    
    prediction = "PHISHING" if final_probs[1] > 0.5 else "SAFE"
    confidence = max(final_probs) * 100
    
    return {
        'prediction': prediction,
        'confidence': confidence,
        'url_weight': expert_weights[0, 0].item() * 100,
        'text_weight': expert_weights[0, 1].item() * 100,
        'url_prediction': 'PHISHING' if url_probs[1] > 0.5 else 'SAFE',
        'text_prediction': 'PHISHING' if text_probs[1] > 0.5 else 'SAFE',
        'url_probs': url_probs,
        'text_probs': text_probs
    }

# ============================================================================
# STEP 7: Testing Interface
# ============================================================================

def test_sample(input_text):
    """
    Test the MoE system with automatic URL/text detection.
    
    Args:
        input_text (str): Input message (may contain text and/or URL)
        
    Returns:
        dict: Complete prediction results
    """
    
    # Automatic URL Detection
    url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
    urls = re.findall(url_pattern, input_text)
    
    if urls:
        url = urls[0]
        text = re.sub(url_pattern, '', input_text).strip()
    else:
        url = ""
        text = input_text.strip()
    
    # Perform Prediction
    results = predict_with_trained_model(text, url)
    
    # Display Results
    print("=" * 70)
    print("PREDICTION RESULTS - Mixture of Experts with Gating Network")
    print("=" * 70)
    
    if text:
        print(f"Text Input: {text[:80]}..." if len(text) > 80 else f"Text Input: {text}")
    if url:
        print(f"URL Input: {url}")
    
    print("\n" + "-" * 70)
    print("Learned Expert Weights (Gating Network Output):")
    print(f"  URL Expert Weight:  {results['url_weight']:.2f}%")
    print(f"  Text Expert Weight: {results['text_weight']:.2f}%")
    
    print("\nIndividual Expert Predictions:")
    print(f"  URL Expert:  {results['url_prediction']} (Confidence: {max(results['url_probs'])*100:.2f}%)")
    print(f"  Text Expert: {results['text_prediction']} (Confidence: {max(results['text_probs'])*100:.2f}%)")
    
    print("-" * 70)
    print(f"FINAL PREDICTION: {results['prediction']}")
    print(f"Overall Confidence: {results['confidence']:.2f}%")
    print("=" * 70)
    print()
    
    return results

# ============================================================================
# STEP 8: Interactive Mode
# ============================================================================

def interactive_mode():
    """
    Interactive testing interface for the MoE phishing detection system.
    Allows continuous testing with user input until exit.
    """
    
    print("\n" + "=" * 70)
    print("INTERACTIVE MODE - Mixture of Experts Phishing Detection")
    print("=" * 70)
    print("\nInstructions:")
    print("  - Enter any message text, URL, or combination")
    print("  - Type 'exit' or 'quit' to end the session")
    print("  - Type 'sample' to test with predefined examples")
    print("=" * 70)
    
    # Predefined sample messages
    samples = [
        "URGENT! Your account has been suspended. Verify now at http://fake-bank.com",
        "Congratulations! You won $1000! Claim here: http://prize-claim.tk",
        "Hey, are we still meeting for lunch tomorrow?",
        "http://paypa1-secure-login.com/verify",
        "Please review the attached document and send feedback.",
        "SECURITY ALERT: Click here to confirm your identity immediately!",
        "Meeting notes from today's discussion are available on Drive.",
        "Limited time offer! Act now to claim your bonus!"
    ]
    
    while True:
        print("\n" + "-" * 70)
        user_input = input("\nEnter message to analyze (or 'exit'/'sample'): ").strip()
        
        # Exit condition
        if user_input.lower() in ['exit', 'quit', 'q']:
            print("\n" + "=" * 70)
            print("SESSION ENDED - Thank you for using the MoE Detection System")
            print("=" * 70 + "\n")
            break
        
        # Sample mode
        if user_input.lower() == 'sample':
            print("\n" + "=" * 70)
            print("PREDEFINED SAMPLE MESSAGES")
            print("=" * 70)
            for i, sample in enumerate(samples, 1):
                print(f"{i}. {sample[:65]}...")
            
            try:
                choice = input("\nSelect sample number (1-{}): ".format(len(samples))).strip()
                choice_idx = int(choice) - 1
                
                if 0 <= choice_idx < len(samples):
                    user_input = samples[choice_idx]
                    print(f"\nSelected: {user_input}\n")
                else:
                    print("\nInvalid selection. Returning to main menu.")
                    continue
            except (ValueError, IndexError):
                print("\nInvalid input. Returning to main menu.")
                continue
        
        # Empty input handling
        if not user_input:
            print("\nError: No input provided. Please enter a message to analyze.")
            continue
        
        # Perform prediction
        try:
            test_sample(user_input)
        except Exception as e:
            print("\n" + "=" * 70)
            print("ERROR DURING PREDICTION")
            print("=" * 70)
            print(f"Error details: {str(e)}")
            print("Please try again with different input.")
            print("=" * 70)

# ============================================================================
# System Ready
# ============================================================================

print("\n" + "=" * 70)
print("SYSTEM READY FOR TESTING")
print("=" * 70)
print("\nAvailable Functions:")
print('  1. test_sample("your message here")  - Single prediction')
print('  2. interactive_mode()                 - Interactive session')
print("\nQuick Start:")
print('  >>> interactive_mode()')
print("=" * 70)

Loading Expert Models...
----------------------------------------------------------------------
Expert 1 (URL-based): Loaded successfully
Expert 2 (Text-based): Loaded successfully

Loading Trained Gating Network...
----------------------------------------------------------------------
Gating Network: Loaded successfully

MoE System Initialization Complete


SYSTEM READY FOR TESTING

Available Functions:
  1. test_sample("your message here")  - Single prediction
  2. interactive_mode()                 - Interactive session

Quick Start:
  >>> interactive_mode()


In [22]:
test_sample("http://paypa1.com")

PREDICTION RESULTS - Mixture of Experts with Gating Network
URL Input: http://paypa1.com

----------------------------------------------------------------------
Learned Expert Weights (Gating Network Output):
  URL Expert Weight:  100.00%
  Text Expert Weight: 0.00%

Individual Expert Predictions:
  URL Expert:  PHISHING (Confidence: 90.72%)
  Text Expert: SAFE (Confidence: 50.00%)
----------------------------------------------------------------------
FINAL PREDICTION: PHISHING
Overall Confidence: 90.72%



{'prediction': 'PHISHING',
 'confidence': 90.7205421515973,
 'url_weight': 100.0,
 'text_weight': 8.159630957016439e-10,
 'url_prediction': 'PHISHING',
 'text_prediction': 'SAFE',
 'url_probs': array([0.09279458, 0.90720542]),
 'text_probs': array([0.5, 0.5])}

In [ ]:
 interactive_mode()


INTERACTIVE MODE - Mixture of Experts Phishing Detection

Instructions:
  - Enter any message text, URL, or combination
  - Type 'exit' or 'quit' to end the session
  - Type 'sample' to test with predefined examples

----------------------------------------------------------------------



Enter message to analyze (or 'exit'/'sample'):  sample



PREDEFINED SAMPLE MESSAGES
1. URGENT! Your account has been suspended. Verify now at http://fak...
2. Congratulations! You won $1000! Claim here: http://prize-claim.tk...
3. Hey, are we still meeting for lunch tomorrow?...
4. http://paypa1-secure-login.com/verify...
5. Please review the attached document and send feedback....
6. SECURITY ALERT: Click here to confirm your identity immediately!...
7. Meeting notes from today's discussion are available on Drive....
8. Limited time offer! Act now to claim your bonus!...



Select sample number (1-8):  1



Selected: URGENT! Your account has been suspended. Verify now at http://fake-bank.com

PREDICTION RESULTS - Mixture of Experts with Gating Network
Text Input: URGENT! Your account has been suspended. Verify now at
URL Input: http://fake-bank.com

----------------------------------------------------------------------
Learned Expert Weights (Gating Network Output):
  URL Expert Weight:  0.00%
  Text Expert Weight: 100.00%

Individual Expert Predictions:
  URL Expert:  PHISHING (Confidence: 96.97%)
  Text Expert: PHISHING (Confidence: 100.00%)
----------------------------------------------------------------------
FINAL PREDICTION: PHISHING
Overall Confidence: 100.00%


----------------------------------------------------------------------



Enter message to analyze (or 'exit'/'sample'):  https://www.youtube.com/watch?v=yIwaEuo6BiI


PREDICTION RESULTS - Mixture of Experts with Gating Network
URL Input: https://www.youtube.com/watch?v=yIwaEuo6BiI

----------------------------------------------------------------------
Learned Expert Weights (Gating Network Output):
  URL Expert Weight:  100.00%
  Text Expert Weight: 0.00%

Individual Expert Predictions:
  URL Expert:  SAFE (Confidence: 99.95%)
  Text Expert: SAFE (Confidence: 50.00%)
----------------------------------------------------------------------
FINAL PREDICTION: SAFE
Overall Confidence: 99.95%


----------------------------------------------------------------------
